# Run Coding Assistant with Cursor

This notebook guides you through running and testing a **Cursor IDE coding assistant** powered by the MaaS unified gateway.

You'll verify that Cursor can:
1. Connect to RHOAI models via MaaS for code generation
2. Access MCP tools (GitHub, docs, search) via the same gateway
3. Use the coding assistant for real development tasks

**Prerequisites:**
- IDE configured with MaaS endpoint and API key (`1_ide_configuration.ipynb` completed)
- Cursor IDE installed ([cursor.com](https://cursor.com))
- MaaS gateway running with models and MCP servers registered

## 1. Verify MaaS Connectivity

Before launching Cursor, confirm the gateway is accessible and models/tools are available.

In [ ]:
import subprocess
import json
import urllib.request
import ssl

result = subprocess.run(
    ["kubectl", "get", "ingresses.config.openshift.io", "cluster",
     "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()
MAAS_HOST = f"https://maas-api.apps.{CLUSTER_DOMAIN}"

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print(f"MaaS Gateway: {MAAS_HOST}")
print("")

# Check models
req = urllib.request.Request(
    f"{MAAS_HOST}/v1/models",
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"}
)
try:
    with urllib.request.urlopen(req, context=ctx) as resp:
        models_data = json.loads(resp.read())
    print("\u2705 Models available:")
    for model in models_data.get("data", []):
        print(f"   {model['id']}  \u2192  {model.get('url', 'N/A')}")
except Exception as e:
    print(f"\u274c Could not reach models API: {e}")
    models_data = {}

print("")

# Check MCP servers
routes_result = subprocess.run(
    ["kubectl", "get", "httproute", "-n", "mcp-servers",
     "-l", "maas.opendatahub.io/managed=true",
     "-o", "jsonpath={range .items[*]}{.metadata.name}\n{end}"],
    capture_output=True, text=True
)

mcp_servers = []
for line in routes_result.stdout.strip().split("\n"):
    if line:
        short_name = line.replace("mcp-route-", "")
        mcp_servers.append(short_name)

if mcp_servers:
    print("\u2705 MCP servers via gateway:")
    for name in mcp_servers:
        print(f"   {name}  \u2192  {MAAS_HOST}/mcp/{name}/sse")
else:
    print("\u26a0\ufe0f  No MCP servers registered with gateway")

print("")
print("\u2705 Ready for Cursor configuration" if models_data.get("data") else "\u26a0\ufe0f  Fix issues above before proceeding")

## 2. Generate Cursor Configuration

Generate the `.cursor/mcp.json` for your project to connect Cursor to MCP tools via MaaS.

In [ ]:
API_KEY_PLACEHOLDER = "sk-oai-YOUR-KEY"

# Build MCP config for Cursor (via MaaS gateway)
cursor_mcp_config = {"mcpServers": {}}

for name in mcp_servers:
    cursor_mcp_config["mcpServers"][name] = {
        "url": f"{MAAS_HOST}/mcp/{name}/sse",
        "headers": {
            "Authorization": f"Bearer {API_KEY_PLACEHOLDER}"
        }
    }

print("=== .cursor/mcp.json ===")
print(json.dumps(cursor_mcp_config, indent=2))
print("")
print("Copy this to your project's .cursor/mcp.json file.")
print(f"Replace '{API_KEY_PLACEHOLDER}' with your actual MaaS API key.")

## 3. Cursor Model Settings

Configure Cursor to use RHOAI models via the MaaS gateway.

### Steps:

1. Open Cursor **Settings** (\u2318+, or Ctrl+,)
2. Navigate to **Models** section
3. Click **Add Model** and configure:

| Setting | Value |
|---------|-------|
| Model Name | *(from cell above, e.g. `qwen-coder-14b`)* |
| Provider | OpenAI Compatible |
| Base URL | *(model URL)/v1* |
| API Key | *(your MaaS API key `sk-oai-...`)* |

4. Enable the model for **Chat** and/or **Agent** mode

In [ ]:
print("=== Cursor Model Configuration ===")
print("")

if models_data.get("data"):
    for model in models_data["data"]:
        print(f"Model:    {model['id']}")
        print(f"Base URL: {model.get('url', 'N/A')}/v1")
        print(f"API Key:  {API_KEY_PLACEHOLDER}")
        print("")
else:
    print("\u26a0\ufe0f  No models detected. Run cell 1 first.")

print("---")
print("\nTip: Use a larger model (e.g. qwen-coder-14b) for Agent mode,")
print("and a smaller model (e.g. qwen-coder-7b) for autocomplete.")

## 4. Test: Code Generation (Chat Mode)

Open Cursor and test basic code generation via the MaaS-connected model.

### Try these prompts in Cursor Chat (\u2318+L):

```
Write a Python FastAPI endpoint that accepts a JSON body with "name" and "email"
fields and returns a greeting message.
```

```
Create a Kubernetes Deployment YAML for an nginx pod with 3 replicas,
resource limits of 256Mi memory and 500m CPU.
```

```
Write a bash script that checks if a given port is in use and
prints the process using it.
```

### Expected behavior:
- Response should stream token-by-token
- Model name shown in the response header should match your RHOAI model
- No errors related to auth or connectivity

## 5. Test: MCP Tools (Agent Mode)

Switch Cursor to **Agent Mode** (\u2318+I) and test MCP tool integration.

### Try these prompts:

**Context7 (Library Docs):**
```
Using @context7, look up the latest FastAPI documentation for dependency injection.
```

**GitHub MCP:**
```
Using the GitHub tool, list the open issues in the kubernetes/kubernetes repository.
```

**Sequential Thinking:**
```
Using sequential thinking, design a microservice architecture for a
user authentication system with OAuth2, JWT tokens, and role-based access.
```

**gh-grep (Code Search):**
```
Using gh-grep, search for how InferenceService is defined in the
opendatahub-io/opendatahub-operator repository.
```

### Expected behavior:
- Cursor should show tool calls in the Agent panel
- Tools connect via MaaS gateway (check the URL matches `maas.<domain>/mcp/...`)
- Results are returned and incorporated into the response

## 6. Test: Inline Edit (\u2318+K)

Test inline code editing capabilities.

### Steps:
1. Create a simple Python file (e.g. `test_edit.py`):

```python
def calculate_total(items):
    total = 0
    for item in items:
        total = total + item["price"] * item["quantity"]
    return total
```

2. Select the function and press **\u2318+K**
3. Type: `Add type hints, use sum() with generator, add docstring`
4. Review the proposed edit and accept/reject

### Expected behavior:
- Model proposes a refactored version with type hints
- Diff view shows changes clearly
- Accepts/rejects apply correctly

## 7. Verify MaaS Usage

After running the tests above, check that requests flowed through the MaaS gateway.

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.apps.${CLUSTER_DOMAIN}"

echo "=== MaaS Usage After Cursor Testing ==="
echo ""

# Check model inference pods (requests should have generated activity)
echo "Model Serving Pods:"
kubectl get pods -A -l component=predictor --no-headers 2>/dev/null | \
    awk '{printf "  %s/%s  Status: %s\n", $1, $2, $4}'

echo ""
echo "MCP Server Pods:"
kubectl get pods -n mcp-servers --no-headers 2>/dev/null | \
    awk '{printf "  %s  Status: %s  Restarts: %s\n", $1, $3, $4}'

echo ""
echo "Gateway Activity (last 5 min):"
# Check Limitador for recent activity
THANOS_HOST="https://thanos-querier-openshift-monitoring.${CLUSTER_DOMAIN}"
RESULT=$(curl -sSk "${THANOS_HOST}/api/v1/query?query=sum(increase(authorized_calls[5m]))" \
  -H "Authorization: Bearer $(oc whoami -t)" 2>/dev/null | \
  python3 -c "import sys,json; d=json.load(sys.stdin); r=d.get('data',{}).get('result',[]); print(r[0]['value'][1] if r else '0')" 2>/dev/null)
echo "  Authorized calls (5m): ${RESULT:-N/A}"

## 8. Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| Cursor shows "Connection failed" | Wrong Base URL or gateway down | Verify URL in cell 1; check `kubectl get gateway -n openshift-ingress` |
| 401 in Cursor output | Invalid API key | Regenerate key via RHOAI Dashboard or `POST /maas-api/v1/api-keys` |
| Model responds but MCP tools fail | MCP config not loaded | Restart Cursor after editing `.cursor/mcp.json` |
| MCP tools show "Unauthorized" | Missing auth header in MCP config | Ensure `headers.Authorization` is set in `.cursor/mcp.json` |
| Slow responses | Model cold start or rate limit | Wait for model pod to scale up; check subscription limits |
| No autocomplete suggestions | Autocomplete model not configured | Add a fast model (2B) as Tab completion in Cursor settings |
| SSL certificate error | Self-signed cert on cluster | Set `"CURSOR_ALLOW_INSECURE": true` or add CA to trust store |

## 9. Tips for Production Use

### Model Selection Strategy

| Task | Recommended Model | Why |
|------|-------------------|-----|
| Tab autocomplete | Qwen2.5-Coder-7B (FP8) | Low latency, small completions |
| Chat / Q&A | Qwen2.5-Coder-7B (FP8) | Balanced quality and speed |
| Agent mode (complex tasks) | Qwen2.5-Coder-14B (FP8) | Best reasoning capability |

### Team Configuration

1. **Commit `.cursor/mcp.json`** to your repo — all team members get MCP tools automatically
2. **Use per-developer API keys** — enables individual rate limiting and audit
3. **Set subscription limits** — prevent runaway usage via MaaSSubscription CRDs

### Security Best Practices

- Store API key in environment variable, not in committed files
- Use short-lived keys for CI/CD and long-lived keys for developer IDEs
- Rotate keys periodically via the MaaS API key management

## Summary

| Test | Status |
|------|--------|
| MaaS connectivity | Run cell 1 |
| Code generation (Chat) | Try prompts in section 4 |
| MCP tools (Agent) | Try prompts in section 5 |
| Inline edit (\u2318+K) | Try steps in section 6 |
| Usage verification | Run cell in section 7 |

### What You've Achieved

\u2705 Cursor IDE connected to self-hosted models on RHOAI  
\u2705 MCP tools accessible through the MaaS unified gateway  
\u2705 Single API key for all AI services (models + tools)  
\u2705 Enterprise-grade auth and rate limiting applied  

## Next Steps

\u2192 `3_maas_advanced.ipynb` \u2014 Fine-tune subscriptions, rate limits, and monitoring